# Imputing numerical data
Rebuilt for scikit-learn 1.9, pandas 3 and Plotly 7. Data: `data/titanic_toy.csv`, four columns of the Titanic data (891 rows). `Family` is siblings/spouses plus parents/children aboard; 5% of `Fare` was deleted at random on purpose.

In [ ]:
# pandas for the table, numpy for numbers, Plotly for charts, scipy for the KDE curves
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import gaussian_kde
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

## 1. Load the data and split it

In [ ]:
# load the data and see how much of each column is missing
df = pd.read_csv("data/titanic_toy.csv")
print(df.shape)
df.isnull().mean()

In [ ]:
# split first: every imputation value below is learned from X_train only
X = df.drop(columns=["Survived"])
y = df["Survived"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)
# copy() gives an independent table, so adding columns raises no SettingWithCopyWarning
X_train = X_train.copy()
X_train.shape, X_test.shape

In [ ]:
# share missing in the training set: about 21% of Age, 5% of Fare
X_train.isnull().mean()

## 2. Mean and median imputation with pandas

In [ ]:
# the four fill values, computed on the training set
mean_age, median_age = X_train["Age"].mean(), X_train["Age"].median()
mean_fare, median_fare = X_train["Fare"].mean(), X_train["Fare"].median()
mean_age, median_age, mean_fare, median_fare

In [ ]:
# fillna replaces every NaN with the given value; we keep the original column to compare
X_train["Age_median"] = X_train["Age"].fillna(median_age)
X_train["Age_mean"] = X_train["Age"].fillna(mean_age)
X_train["Fare_median"] = X_train["Fare"].fillna(median_fare)
X_train["Fare_mean"] = X_train["Fare"].fillna(mean_fare)
X_train.sample(5, random_state=0)

### 2.1 Variance before and after

In [ ]:
# var() skips NaN, so 'Age' is the variance of the known values only
X_train[["Age", "Age_median", "Age_mean", "Fare", "Fare_median", "Fare_mean"]].var()

In [ ]:
# mean imputation shrinks the variance by exactly (known - 1) / (all - 1)
n, k = len(X_train), X_train["Age"].notna().sum()
X_train["Age"].var() * (k - 1) / (n - 1)

### 2.2 Distribution before and after (KDE)

In [ ]:
# draw the density curve of several columns on the same axes
def compare_kde(cols, lo, hi):
    grid = np.linspace(lo, hi, 400)
    fig = go.Figure()
    for c in cols:
        fig.add_trace(go.Scatter(x=grid, y=gaussian_kde(X_train[c].dropna())(grid), name=c))
    fig.update_layout(template="simple_white", xaxis_title="value", yaxis_title="density")
    fig.show()

compare_kde(["Age", "Age_median", "Age_mean"], -10, 85)
compare_kde(["Fare", "Fare_median", "Fare_mean"], -50, 300)

### 2.3 Covariance and correlation before and after

In [ ]:
# cov() and corr() also skip NaN, pair of columns by pair of columns
X_train.cov().loc[["Age", "Age_median", "Age_mean", "Fare", "Fare_median", "Fare_mean"], ["Age", "Fare", "Family"]]

In [ ]:
X_train.corr().loc[["Age", "Age_median", "Age_mean", "Fare", "Fare_median", "Fare_mean"], ["Age", "Fare", "Family"]]

### 2.4 Outliers before and after (box plots)

In [ ]:
# a box plot per column; points beyond the whiskers are outliers
def compare_box(cols):
    fig = go.Figure([go.Box(y=X_train[c], name=c) for c in cols])
    fig.update_layout(template="simple_white", showlegend=False)
    fig.show()

compare_box(["Age", "Age_median", "Age_mean"])
compare_box(["Fare", "Fare_median", "Fare_mean"])

In [ ]:
# count the outliers with the 1.5 IQR rule
def n_outliers(s):
    q1, q3 = s.quantile([0.25, 0.75])
    return ((s < q1 - 1.5 * (q3 - q1)) | (s > q3 + 1.5 * (q3 - q1))).sum()

{c: n_outliers(X_train[c]) for c in ["Age", "Age_median", "Age_mean", "Fare", "Fare_median", "Fare_mean"]}

## 3. Mean and median imputation with scikit-learn

In [ ]:
# a fresh split, so the extra columns above are gone
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)

# median for Age, mean for Fare; Family passes through unchanged
trf = ColumnTransformer([
    ("imputer1", SimpleImputer(strategy="median"), ["Age"]),
    ("imputer2", SimpleImputer(strategy="mean"), ["Fare"]),
], remainder="passthrough", verbose_feature_names_out=False)
# return pandas DataFrames instead of numpy arrays
trf.set_output(transform="pandas")

In [ ]:
# fit learns the median and mean from the training set only
trf.fit(X_train)
trf.named_transformers_["imputer1"].statistics_, trf.named_transformers_["imputer2"].statistics_

In [ ]:
# transform fills both sets with the training values
X_train_imp = trf.transform(X_train)
X_test_imp = trf.transform(X_test)
X_train_imp.isnull().sum().sum(), X_test_imp.isnull().sum().sum()

In [ ]:
X_test_imp.head()

## 4. Arbitrary value imputation with pandas

In [ ]:
X_train = X_train.copy()
# fill with values that never occur: 99 or -1 for Age, 999 or -1 for Fare
X_train["Age_99"] = X_train["Age"].fillna(99)
X_train["Age_minus1"] = X_train["Age"].fillna(-1)
X_train["Fare_999"] = X_train["Fare"].fillna(999)
X_train["Fare_minus1"] = X_train["Fare"].fillna(-1)
X_train[["Age", "Age_99", "Age_minus1", "Fare", "Fare_999", "Fare_minus1"]].var()

In [ ]:
compare_kde(["Age", "Age_99", "Age_minus1"], -20, 120)
compare_kde(["Fare", "Fare_999", "Fare_minus1"], -100, 1100)

In [ ]:
X_train.cov().loc[["Age", "Age_99", "Age_minus1", "Fare", "Fare_999", "Fare_minus1"], ["Age", "Fare", "Family"]]

In [ ]:
X_train.corr().loc[["Age", "Age_99", "Age_minus1", "Fare", "Fare_999", "Fare_minus1"], ["Age", "Fare", "Family"]]

## 5. Arbitrary value imputation with scikit-learn

In [ ]:
# strategy="constant" fills with fill_value
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)
trf = ColumnTransformer([
    ("imputer1", SimpleImputer(strategy="constant", fill_value=99), ["Age"]),
    ("imputer2", SimpleImputer(strategy="constant", fill_value=999), ["Fare"]),
], remainder="passthrough", verbose_feature_names_out=False).set_output(transform="pandas")
trf.fit(X_train)
trf.named_transformers_["imputer1"].statistics_, trf.named_transformers_["imputer2"].statistics_

In [ ]:
X_train_imp = trf.transform(X_train)
X_test_imp = trf.transform(X_test)
X_train_imp.tail()

## 6. End of distribution imputation

In [ ]:
# Age is close to normal: use mean + 3 standard deviations (training set)
age_end = X_train["Age"].mean() + 3 * X_train["Age"].std()
# Fare is skewed: use Q3 + 1.5 IQR (training set)
q1, q3 = X_train["Fare"].quantile([0.25, 0.75])
fare_end = q3 + 1.5 * (q3 - q1)
age_end, fare_end

In [ ]:
# the same constant strategy, with the end values as fill_value
trf = ColumnTransformer([
    ("age", SimpleImputer(strategy="constant", fill_value=age_end), ["Age"]),
    ("fare", SimpleImputer(strategy="constant", fill_value=fare_end), ["Fare"]),
], remainder="passthrough", verbose_feature_names_out=False).set_output(transform="pandas")
X_train_imp = trf.fit_transform(X_train)
X_test_imp = trf.transform(X_test)
pd.DataFrame({"before": X_train[["Age", "Fare"]].var(), "after": X_train_imp[["Age", "Fare"]].var()})

In [ ]:
# compare the shapes
X_train_end = X_train.assign(Age_end=X_train_imp["Age"], Fare_end=X_train_imp["Fare"])
grid = np.linspace(-10, 90, 400)
fig = go.Figure([go.Scatter(x=grid, y=gaussian_kde(X_train_end[c].dropna())(grid), name=c) for c in ["Age", "Age_end"]])
fig.update_layout(template="simple_white", xaxis_title="Age", yaxis_title="density")
fig.show()